# Gold Hiring Activity Mart

**Purpose**: Hiring trends and analysis across all sectors.

**Target Table**: `workspace.gold.gold_hiring_activity`

**Grain**: One row per date per sector with aggregate hiring metrics

**Sector Support**: Multi-sector enabled via sector_sk dimension

**Key Metrics**:
- Active jobs, new postings, closures
- Top roles and locations
- Salary statistics
- Remote vs onsite distribution
- Temporal trends

**Data Sources**:
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.dim_job`
- `workspace.warehouse.dim_sector`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_hiring_activity
USING DELTA
COMMENT 'Hiring trends and analysis across all sectors'
AS

WITH base_metrics AS (
  SELECT
    f.posting_date_sk AS hiring_date_sk,
    f.sector_sk,
    f.role_sk,
    f.location_sk,
    
    -- MEASURES
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS closed_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) - 
      COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS net_change,
    
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.company_sk END) AS unique_companies,
    
    -- Remote work metrics
    COUNT(CASE WHEN f.active_flag = TRUE AND j.remote_type = 'REMOTE' THEN 1 END) AS remote_jobs,
    COUNT(CASE WHEN f.active_flag = TRUE AND j.remote_type = 'ONSITE' THEN 1 END) AS onsite_jobs,
    
    -- Salary metrics
    AVG(CASE WHEN f.active_flag = TRUE THEN j.salary_midpoint END) AS avg_salary,
    PERCENTILE(CASE WHEN f.active_flag = TRUE THEN j.salary_midpoint END, 0.5) AS median_salary
    
  FROM workspace.warehouse.fact_job_postings f
  LEFT JOIN workspace.warehouse.dim_job j ON f.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND f.sector_sk IS NOT NULL  -- Ensure valid sector assignment
  GROUP BY f.posting_date_sk, f.sector_sk, f.role_sk, f.location_sk
),

date_sector_agg AS (
  SELECT
    bm.hiring_date_sk,
    bm.sector_sk,
    
    -- Aggregate measures
    SUM(bm.active_jobs) AS total_active_jobs,
    SUM(bm.new_jobs) AS total_new_jobs,
    SUM(bm.closed_jobs) AS total_closed_jobs,
    SUM(bm.net_change) AS total_net_change,
    
    -- Distinct counts
    COUNT(DISTINCT bm.role_sk) AS unique_roles,
    COUNT(DISTINCT bm.location_sk) AS unique_locations,
    SUM(bm.unique_companies) AS total_companies,
    
    -- Remote work
    SUM(bm.remote_jobs) AS total_remote_jobs,
    SUM(bm.onsite_jobs) AS total_onsite_jobs,
    
    -- Salary stats
    AVG(bm.avg_salary) AS avg_salary,
    AVG(bm.median_salary) AS median_salary,
    
    -- Top role (most jobs)
    FIRST(bm.role_sk, true) OVER (
      PARTITION BY bm.hiring_date_sk, bm.sector_sk
      ORDER BY bm.active_jobs DESC
    ) AS top_role_sk,
    
    -- Top location (most jobs)
    FIRST(bm.location_sk, true) OVER (
      PARTITION BY bm.hiring_date_sk, bm.sector_sk
      ORDER BY bm.active_jobs DESC
    ) AS top_location_sk
    
  FROM base_metrics bm
  GROUP BY bm.hiring_date_sk, bm.sector_sk
),

temporal_enriched AS (
  SELECT
    dsa.*,
    
    -- Prior period comparisons
    LAG(dsa.total_active_jobs, 7) OVER (
      PARTITION BY dsa.sector_sk
      ORDER BY dsa.hiring_date_sk
    ) AS prev_week_active_jobs,
    
    LAG(dsa.total_active_jobs, 30) OVER (
      PARTITION BY dsa.sector_sk
      ORDER BY dsa.hiring_date_sk
    ) AS prev_month_active_jobs,
    
    -- Rolling averages
    AVG(dsa.total_new_jobs) OVER (
      PARTITION BY dsa.sector_sk
      ORDER BY dsa.hiring_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_new_jobs,
    
    AVG(dsa.total_active_jobs) OVER (
      PARTITION BY dsa.sector_sk
      ORDER BY dsa.hiring_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_avg_active_jobs
    
  FROM date_sector_agg dsa
)

SELECT
  -- DIMENSIONS (sector_sk first for partitioning)
  te.sector_sk,
  te.hiring_date_sk,
  
  -- MEASURES
  te.total_active_jobs AS total_jobs,
  te.total_new_jobs AS new_jobs,
  te.total_closed_jobs AS closed_jobs,
  te.total_net_change AS net_change,
  te.unique_roles,
  te.unique_locations,
  te.total_companies,
  te.total_remote_jobs,
  te.total_onsite_jobs,
  
  -- Top dimensions
  te.top_role_sk,
  te.top_location_sk,
  
  -- Salary stats
  CAST(te.avg_salary AS DECIMAL(15,2)) AS avg_salary,
  CAST(te.median_salary AS DECIMAL(15,2)) AS median_salary,
  
  -- TEMPORAL METRICS
  CAST(te.rolling_7day_avg_new_jobs AS DECIMAL(10,2)) AS rolling_7day_avg_new_jobs,
  CAST(te.rolling_30day_avg_active_jobs AS DECIMAL(10,2)) AS rolling_30day_avg_active_jobs,
  
  -- DERIVED: Percent changes
  CASE 
    WHEN te.prev_week_active_jobs > 0 
    THEN CAST((te.total_active_jobs - te.prev_week_active_jobs) AS DECIMAL(10,4)) / te.prev_week_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_week,
  
  CASE 
    WHEN te.prev_month_active_jobs > 0 
    THEN CAST((te.total_active_jobs - te.prev_month_active_jobs) AS DECIMAL(10,4)) / te.prev_month_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_month,
  
  -- DERIVED: Remote work percentage
  CASE 
    WHEN te.total_active_jobs > 0 
    THEN CAST(te.total_remote_jobs AS DECIMAL(10,4)) / te.total_active_jobs
    ELSE NULL 
  END AS remote_job_pct,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS updated_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
ORDER BY te.sector_sk, te.hiring_date_sk DESC;


In [0]:
%sql
-- Validation: Recent hiring trends by sector
SELECT
  s.sector_name,
  s.sector_family,
  COUNT(DISTINCT gha.hiring_date_sk) AS days_active,
  SUM(gha.total_jobs) AS total_active_jobs,
  SUM(gha.new_jobs) AS total_new_jobs,
  ROUND(AVG(gha.avg_salary), 2) AS avg_salary,
  ROUND(AVG(gha.remote_job_pct), 4) AS avg_remote_pct
FROM workspace.gold.gold_hiring_activity gha
INNER JOIN workspace.warehouse.dim_sector s ON gha.sector_sk = s.sector_sk
WHERE gha.hiring_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 30), 'yyyyMMdd') AS INT)
GROUP BY s.sector_name, s.sector_family
ORDER BY total_active_jobs DESC
LIMIT 20;
